In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
%cd ..

In [ ]:
import numpy as np
import healpy as hp
import h5py
from DVA_compute import aberration_sh_rotator, planar_orbit_velocity_direction
from scipy.spatial.transform import Rotation

import Rotation_Functions as rf
import ObservingField
import Make_ell2_GW_Background as GWB

In [ ]:
#Generate Roman fields, populated with random stars.

num_stars = 1024
stars_per_side = 32
test_field0 = ObservingField.ObservingField(np.array([90.1 + 0.53/2.,0])*np.pi/180., 0.53*np.pi/180.)
#test_field0.generate_random_stars_uniform(num_stars=num_stars)
test_field0.generate_stars_uniform(stars_per_side=stars_per_side)

test_field1 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0.53/2.,360. -3*.53])*np.pi/180., 0.53*np.pi/180.)
#test_field1.generate_random_stars_uniform(num_stars=num_stars)
test_field1.generate_stars_uniform(stars_per_side=stars_per_side)

test_field2 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0.53/2.,360. -2*.53])*np.pi/180., 0.53*np.pi/180.)
#test_field2.generate_random_stars_uniform(num_stars=num_stars)
test_field2.generate_stars_uniform(stars_per_side=stars_per_side)

test_field3 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0.53/2.,360. -1*.53])*np.pi/180., 0.53*np.pi/180.)
#test_field3.generate_random_stars_uniform(num_stars=num_stars)
test_field3.generate_stars_uniform(stars_per_side=stars_per_side)

test_field4 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0.53/2.,360.])*np.pi/180., 0.53*np.pi/180.)
#test_field4.generate_random_stars_uniform(num_stars=num_stars)
test_field4.generate_stars_uniform(stars_per_side=stars_per_side)

test_field5 = ObservingField.ObservingField(np.array([90.1 + 0.6 + 0*50 + 0.53/2.,360. +.53])*np.pi/180., 0.53*np.pi/180.)
#test_field5.generate_random_stars_uniform(num_stars=num_stars)
test_field5.generate_stars_uniform(stars_per_side=stars_per_side)

fields = [test_field0, test_field1, test_field2, test_field3, test_field4, test_field5]

In [ ]:
#Plot starting positions of guide stars.
for field in fields:
    for ind in range(field.original_guide_stars_xyz.shape[0]):
        plt.scatter(field.original_guide_stars_xyz[ind,1], 
                field.original_guide_stars_xyz[ind,2])
plt.xlabel(r'$\hat{y}$')
plt.ylabel(r'$\hat{z}$')
plt.title('Field guide stars, Galactic coordinates.')
plt.show()

In [ ]:
#Generate ell=2 vector spherical harmonics at Roman star positions.

gw_sim_field_stars = []
gw_sim_guide_stars = []
for field in fields:
    gw_sim_field_stars.append(GWB.ell2_vector_harmonics(field.stars_original_positions_theta_phi))
    gw_sim_guide_stars.append(GWB.ell2_vector_harmonics(field.original_guide_stars_theta_phi))

In [ ]:
#Make list of observation times.
#This notebook will pretend there is full time observing on 6 Roman fields over 5 years.
#Each field is re-observed every 15 minutes.
mission_time_seconds = 5*365*24*60*60
full_field_cadence_seconds = 15*60

observing_times_seconds = np.arange(0, 
            mission_time_seconds + full_field_cadence_seconds, 
                                    full_field_cadence_seconds)
observing_times_years = observing_times_seconds/(365*24*60*60)

print(observing_times_years)
print(observing_times_years.shape)

In [ ]:
#Compute DVA over a circular orbit.
orbit_object = planar_orbit_velocity_direction()
orbit_object.compute_circular_orbit_theta_phi(observing_times_years, 1, 0)
boost_magnitude = 10**-4

plt.plot(observing_times_years,
         orbit_object.circular_orbit_thetas*180./np.pi, label = 'Theta')
plt.plot(observing_times_years,
         orbit_object.circular_orbit_phis*180./np.pi, label = 'Phi')
plt.title('Orbital velocity direction, Galactic coordinates.')
plt.xlabel('Time (years)')
plt.ylabel('Angle (degrees)')
plt.legend()
plt.show()

In [ ]:
#Model the (Black Hole Binary) Gravitatinal Wave Background.

gwb_object = GWB.gw_background(10**-14, 1./(2*full_field_cadence_seconds), 
                               1./(mission_time_seconds))
gwb_object.predict_full_sky_astrometric_gw_variance()
print(gwb_object.frequencies.size)

In [ ]:
#Plot Graviational Wave Background amplitude.

plt.plot(gwb_object.frequencies[1:], gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal')
plt.hlines(gwb_object.full_sky_astrometric_gw_variance_dn_freq**0.5, 
           gwb_object.frequencies[1],
           gwb_object.frequencies[-1], label = 'Error, full sky, unbinned',
          color = 'orange')
plt.hlines((gwb_object.full_sky_astrometric_gw_variance_dn_freq/100)**0.5, 
           gwb_object.frequencies[1],
           gwb_object.frequencies[-1], label = 'Error, full sky, binnedx100',
          color = 'green')
plt.xscale('log')
plt.yscale('log')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.xlabel('Frequencies (Hz)')
plt.legend()
plt.title(r'RMS Deflection from BHBB in $\Delta f =$' + '{:0.1e}'.format(gwb_object.delta_f))
plt.show()

plt.plot(gwb_object.frequencies[1:], gwb_object.hc[1:], label = 'Signal')
plt.plot(gwb_object.frequencies[1:], 
         gwb_object.full_sky_astrometric_gw_variance_hc[1:]**0.5,
        label = 'Error, full sky, unbinned')
plt.plot(gwb_object.frequencies[1:], 
         (gwb_object.full_sky_astrometric_gw_variance_hc[1:]/10)**0.5,
        label = 'Error, full sky, binnedx100')
plt.xscale('log')
plt.yscale('log')
plt.ylabel(r'$h_c$ (radians)')
plt.xlabel('Frequencies (Hz)')
plt.title(r'Characteristic Strain')
plt.legend()
plt.show()

In [ ]:
#Draw a realization of the gravitational wave background.
#Model the background via 10 ell=2 E/B vector spherical harmonics.

GW_alms_freq_space = gwb_object.realize_spherical_harmonic_amplitudes_freq_space()
for ind in range(10):
    plt.plot(gwb_object.ell2_times_seconds, 
             gwb_object.ell2_time_amplitudes[ind,:])
plt.xlabel('Time (seconds)')
plt.title(r'$\ell=2$ spherical harmonic amplitudes, BHBB')

In [ ]:
#Now simulate observation of GW background.
#Simulate with and without noise.
#Also simulate with and without DVA.

#Dimension is (# of fields, # of gradient modes, # of time samples).
magnitudes_no_DVA_no_noise_no_DVA_fitting = np.zeros((6, 4, 
                                                      observing_times_seconds.size))
magnitudes_no_DVA_no_noise_with_DVA_fitting = np.zeros((6, 4, 
                                                      observing_times_seconds.size))

magnitudes_with_DVA_no_noise_no_DVA_fitting = np.zeros((6, 4, 
                                                      observing_times_seconds.size))
magnitudes_with_DVA_no_noise_with_DVA_fitting = np.zeros((6, 4, 
                                                      observing_times_seconds.size))

magnitudes_with_DVA_with_noise_no_DVA_fitting = np.zeros((6, 4, 
                                                      observing_times_seconds.size))
magnitudes_with_DVA_with_noise_with_DVA_fitting = np.zeros((6, 4, 
                                                      observing_times_seconds.size))

magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting = np.zeros((6, 4, 
                                                      observing_times_seconds.size))
magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting = np.zeros((6, 4, 
                                                      observing_times_seconds.size))

noise_level_arcmin_per_star = 1.e-3
#noise_level_arcmin_per_star = 1.e-5
noise_level_radians_per_star = noise_level_arcmin_per_star/(3600.)*np.pi/180.

for ind_t in range(observing_times_seconds.size):
    if ind_t%10000 == 0:
        print(ind_t)
    for ind in range(6):
        gw_sim_guide_stars[ind].generate_ell2_gw_signal(gwb_object.ell2_time_amplitudes[:5, ind_t], 
                                                        gwb_object.ell2_time_amplitudes[5:, ind_t])
        gw_sim_field_stars[ind].generate_ell2_gw_signal(gwb_object.ell2_time_amplitudes[:5, ind_t], 
                                                        gwb_object.ell2_time_amplitudes[5:, ind_t])
        fields[ind].compute_aberration_from_original_positions(boost_magnitude, 
                                                               orbit_object.circular_orbit_thetas[ind_t],
                                                             orbit_object.circular_orbit_phis[ind_t], 
                                                               boost_order = 3)

        
        fields[ind].perturb_original_guide_stars(gw_sim_guide_stars[ind].gw_signal_ell2_cartesian_vector, 
                                                add_noise = False)
        fields[ind].perturb_original_field_stars(gw_sim_field_stars[ind].gw_signal_ell2_cartesian_vector, 
                                                 add_noise = False)
        fields[ind].project_onto_gradient_modes()
        magnitudes_no_DVA_no_noise_no_DVA_fitting[ind,:,ind_t] = fields[ind].diff_proj_to_grad
        magnitudes_no_DVA_no_noise_with_DVA_fitting[ind,:,ind_t] = fields[ind].diff_minus_DVA_rot_fit_proj_to_grad



        fields[ind].perturb_original_guide_stars(fields[ind].DVA_perturbation_guide_stars_xyz + gw_sim_guide_stars[ind].gw_signal_ell2_cartesian_vector,
                                                add_noise = False)
        fields[ind].perturb_original_field_stars(fields[ind].DVA_perturbation_field_stars_xyz + gw_sim_field_stars[ind].gw_signal_ell2_cartesian_vector, 
                                                 add_noise = False)
        fields[ind].project_onto_gradient_modes()
        magnitudes_with_DVA_no_noise_no_DVA_fitting[ind,:,ind_t] = fields[ind].diff_proj_to_grad
        magnitudes_with_DVA_no_noise_with_DVA_fitting[ind,:,ind_t] = fields[ind].diff_minus_DVA_rot_fit_proj_to_grad


        
        fields[ind].perturb_original_guide_stars(fields[ind].DVA_perturbation_guide_stars_xyz + gw_sim_guide_stars[ind].gw_signal_ell2_cartesian_vector, add_noise = True,
                                                noise_level=noise_level_radians_per_star)
        fields[ind].perturb_original_field_stars(fields[ind].DVA_perturbation_field_stars_xyz + gw_sim_field_stars[ind].gw_signal_ell2_cartesian_vector, add_noise = True, 
                                                 noise_level=noise_level_radians_per_star/(100.))
        fields[ind].project_onto_gradient_modes()
        magnitudes_with_DVA_with_noise_no_DVA_fitting[ind,:,ind_t] = fields[ind].diff_proj_to_grad
        magnitudes_with_DVA_with_noise_with_DVA_fitting[ind,:,ind_t] = fields[ind].diff_minus_DVA_rot_fit_proj_to_grad


        fields[ind].perturb_original_guide_stars(fields[ind].DVA_perturbation_guide_stars_xyz, add_noise = False)
        fields[ind].perturb_original_field_stars(fields[ind].DVA_perturbation_field_stars_xyz, add_noise = False)
        fields[ind].project_onto_gradient_modes()
        magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting[ind,:,ind_t] = fields[ind].diff_proj_to_grad
        magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting[ind,:,ind_t] = fields[ind].diff_minus_DVA_rot_fit_proj_to_grad



        

In [ ]:
#Plot results with no simulated DVA and no simulated noise.

labels = ['XX', 'XY', 'YY', 'YX']
for ind_field in range(6):
    for ind in range(4):
        plt.plot(observing_times_years, 
            num_stars**-0.5*magnitudes_no_DVA_no_noise_no_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.title('Gradient Modes, no simulated DVA or noise, no fitted DVA removal, field ' + str(ind_field))
    plt.show()
    

In [ ]:
#Plot results with no simulated DVA and no simulated noise.

labels = ['XX', 'XY', 'YY', 'YX']
for ind_field in range(6):
    for ind in range(4):
        plt.plot(observing_times_years,
        num_stars**-0.5*magnitudes_no_DVA_no_noise_with_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.title('Gradient Modes, no simulated DVA or noise, fitted DVA removal, field ' + str(ind_field))
    plt.show()

In [ ]:
#Plot results with simulated DVA and no simulated noise.

for ind_field in range(6):
    for ind in range(4):
        plt.plot(observing_times_years,
        num_stars**-0.5*magnitudes_with_DVA_no_noise_no_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.title('Gradient Modes, with simulated DVA but no noise, no fitted DVA removal, field ' + str(ind_field))
    plt.show()

In [ ]:
for ind_field in range(6):
    for ind in range(4):
        plt.plot(observing_times_years,
        num_stars**-0.5*magnitudes_with_DVA_no_noise_with_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.title('Gradient Modes, with simulated DVA but no noise, with fitted DVA removal, field ' + str(ind_field))
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.show()

In [ ]:
#Plot results with simulated DVA and simulated noise.

for ind_field in range(6):
    for ind in range(4):
        plt.plot(observing_times_years,
        num_stars**-0.5*magnitudes_with_DVA_with_noise_no_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.title('Gradient Modes, with simulated DVA and noise, no fitted DVA removal, field ' + str(ind_field))
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.show()

In [ ]:
#Plot results with simulated DVA and simulated noise.

for ind_field in range(6):
    for ind in range(4):
        plt.plot(observing_times_years,
    num_stars**-0.5*magnitudes_with_DVA_with_noise_with_DVA_fitting[ind_field, ind,:],
                label = labels[ind])
    plt.title('Gradient Modes, with simulated DVA and noise, with fitted DVA removal, field ' + str(ind_field))
    plt.legend()
    plt.xlabel('Time (years)')
    plt.ylabel('RMS deflection (radians)')
    plt.show()

In [ ]:
nan_indices = np.where(np.isnan(magnitudes_with_DVA_with_noise_with_DVA_fitting))
for ind in range(nan_indices[0].size):
    print(magnitudes_with_DVA_with_noise_with_DVA_fitting[nan_indices[0][ind],nan_indices[1][ind],nan_indices[2][ind]])
    magnitudes_with_DVA_with_noise_with_DVA_fitting[nan_indices[0][ind],nan_indices[1][ind],nan_indices[2][ind]] = magnitudes_with_DVA_with_noise_with_DVA_fitting[nan_indices[0][ind],nan_indices[1][ind],nan_indices[2][ind]+1]
#print(nan_indices)

In [ ]:
#Compute Fourier transform of gradient mode measurements.

n = magnitudes_no_DVA_no_noise_no_DVA_fitting.shape[-1]
measured_freqs = np.fft.rfftfreq(n)*2*gwb_object.f_nyquist

magnitudes_no_DVA_no_noise_no_DVA_fitting_FT = (1./n)*np.fft.rfft(magnitudes_no_DVA_no_noise_no_DVA_fitting)
magnitudes_no_DVA_no_noise_with_DVA_fitting_FT = (1./n)*np.fft.rfft(magnitudes_no_DVA_no_noise_with_DVA_fitting)

magnitudes_with_DVA_no_noise_no_DVA_fitting_FT = (1./n)*np.fft.rfft(magnitudes_with_DVA_no_noise_no_DVA_fitting)
magnitudes_with_DVA_no_noise_with_DVA_fitting_FT = (1./n)*np.fft.rfft(magnitudes_with_DVA_no_noise_with_DVA_fitting)

magnitudes_with_DVA_with_noise_no_DVA_fitting_FT = (1./n)*np.fft.rfft(magnitudes_with_DVA_with_noise_no_DVA_fitting)
magnitudes_with_DVA_with_noise_with_DVA_fitting_FT = (1./n)*np.fft.rfft(magnitudes_with_DVA_with_noise_with_DVA_fitting)

magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting_FT = (1./n)*np.fft.rfft(magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting)
magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting_FT = (1./n)*np.fft.rfft(magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting)



In [ ]:
#Use a very simple, almost optimal, technique to avoid astrometric noise bias.
#Each field is viewing basically the same signal, separated by just 3 minutes and a small angle.
#But the astrometric noise between each field will of course be uncorrelated.
#Use the sum of all cross_powers between fields. 
#For 6 fields, this loses ~1/6 of the data but avoids any white noise bias.

def get_cross_psd(field_mode_ffts):
    num_fields = field_mode_ffts.shape[0]
    output_ps = np.zeros((field_mode_ffts.shape[1], field_mode_ffts.shape[2]))
    for ind1 in range(num_fields):
        for ind2 in range(num_fields):
            if ind2 != ind1:
                output_ps += np.real(field_mode_ffts[ind1,:,:]*np.conjugate(field_mode_ffts[ind2,:,:]))
    output_ps /= (num_fields**2 - num_fields)
    return output_ps

PSD_field_cross_no_DVA_no_noise_no_DVA_fitting = get_cross_psd(magnitudes_no_DVA_no_noise_no_DVA_fitting_FT)
PSD_field_cross_no_DVA_no_noise_with_DVA_fitting = get_cross_psd(magnitudes_no_DVA_no_noise_with_DVA_fitting_FT)

PSD_field_cross_with_DVA_no_noise_no_DVA_fitting = get_cross_psd(magnitudes_with_DVA_no_noise_no_DVA_fitting_FT)
PSD_field_cross_with_DVA_no_noise_with_DVA_fitting = get_cross_psd(magnitudes_with_DVA_no_noise_with_DVA_fitting_FT)

PSD_field_cross_with_DVA_with_noise_no_DVA_fitting = get_cross_psd(magnitudes_with_DVA_with_noise_no_DVA_fitting_FT)
PSD_field_cross_with_DVA_with_noise_with_DVA_fitting = get_cross_psd(magnitudes_with_DVA_with_noise_with_DVA_fitting_FT)

PSD_field_cross_with_DVA_no_noise_no_GW_no_DVA_fitting = get_cross_psd(magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting_FT)
PSD_field_cross_with_DVA_no_noise_no_GW_with_DVA_fitting = get_cross_psd(magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting_FT)


In [ ]:

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*np.abs(magnitudes_no_DVA_no_noise_no_DVA_fitting_FT[1,ind,:])**2,
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 PSD, no noise, no DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*np.abs(magnitudes_no_DVA_no_noise_with_DVA_fitting_FT[1,ind,:])**2,
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 PSD, no noise, no DVA, with DVA fitting.')
plt.show()

In [ ]:

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*PSD_field_cross_no_DVA_no_noise_no_DVA_fitting[ind,:],
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Combined Field Cross-PSD, no noise, no DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*PSD_field_cross_no_DVA_no_noise_with_DVA_fitting[ind,:],
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Combined Field Cross-PSD, no noise, no DVA, with DVA fitting.')
plt.show()

In [ ]:

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*np.abs(magnitudes_with_DVA_no_noise_no_DVA_fitting_FT[1,ind,:])**2,
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 PSD, no noise, with DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*np.abs(magnitudes_with_DVA_no_noise_with_DVA_fitting_FT[1,ind,:])**2,
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 PSD, no noise, with DVA, with DVA fitting.')
plt.show()

In [ ]:
for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*PSD_field_cross_with_DVA_no_noise_no_DVA_fitting[ind,:],
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Combined Field Cross-PSD, no noise, with DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*PSD_field_cross_with_DVA_no_noise_with_DVA_fitting[ind,:],
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Combined Field Cross-PSD, no noise, with DVA, with DVA fitting.')
plt.show()

In [ ]:

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*np.abs(magnitudes_with_DVA_no_noise_no_GW_no_DVA_fitting_FT[1,ind,:])**2,
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 PSD, no noise, with DVA, no GW, no DVA fitting.')
plt.show()

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*np.abs(magnitudes_with_DVA_no_noise_no_GW_with_DVA_fitting_FT[1,ind,:])**2,
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 PSD, no noise, with DVA, no GW, with DVA fitting.')
plt.show()

In [ ]:
for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*PSD_field_cross_with_DVA_no_noise_no_GW_no_DVA_fitting[ind,:],
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Combined Field Cross-PSD, no noise, with DVA, no GW, no DVA fitting.')
plt.show()

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*PSD_field_cross_with_DVA_no_noise_no_GW_with_DVA_fitting[ind,:],
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Combined Field Cross-PSD, no noise, with DVA, no GW, with DVA fitting.')
plt.show()

In [ ]:

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*np.abs(magnitudes_with_DVA_with_noise_no_DVA_fitting_FT[1,ind,:])**2,
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 PSD, with noise, with DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*np.abs(magnitudes_with_DVA_with_noise_with_DVA_fitting_FT[1,ind,:])**2,
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Field 1 PSD, with noise, with DVA, with DVA fitting.')
plt.show()



In [ ]:

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*PSD_field_cross_with_DVA_with_noise_no_DVA_fitting[ind,:],
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('symlog', linthresh=10**-40)
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Combined Field Cross-PSD, with noise, with DVA, no DVA fitting.')
plt.show()

for ind in range(4):
    plt.plot(measured_freqs, num_stars**-1*PSD_field_cross_with_DVA_with_noise_with_DVA_fitting[ind,:],
            label = labels[ind])
plt.plot(gwb_object.frequencies[1:], 2**-4*10**-4*gwb_object.dn_theta_freq_variance[1:],
        label = 'Signal prediction')
plt.legend()
plt.xscale('log')
plt.yscale('symlog', linthresh=10**-40)
plt.xlabel('Frequency (Hz)')
plt.ylabel(r'$\theta_{rms}^2$ (radians$^2$)')
plt.title('Combined Field Cross-PSD, with noise, with DVA, with DVA fitting.')
plt.show()

In [ ]:

analytical_noise_prediction = gwb_object.full_sky_astrometric_gw_variance_dn_freq**0.5
print('Analytical noise prediction: ' + str(analytical_noise_prediction))

predicted_signal_to_noise_no_mean_subtraction = (np.sum((gwb_object.dn_theta_freq_variance)**2)*analytical_noise_prediction**-2)**0.5
print('Analytical S/N prediction w/o mean subtraction: ' + str(predicted_signal_to_noise_no_mean_subtraction))

measured_noise_level = np.mean(np.std(num_stars**-1*PSD_field_cross_with_DVA_with_noise_with_DVA_fitting, 
             axis = -1))/2.
print('Measured noise level: ' + str(measured_noise_level))

